In [26]:
# Import the pandas library for data manipulation
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
dataset=pd.read_csv("/content/drive/MyDrive/HopeAI/ml_regression_dataset/insurance_pre.csv")
# Display the loaded dataset
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [28]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [29]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [30]:
independent= dataset[['age', 'bmi', 'children', 'sex_male','smoker_yes']]
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [31]:
dependent= dataset[["charges"]]
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [32]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(independent,dependent,test_size=0.30,random_state=0)

In [33]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [34]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
param_grid = {
    'criterion': ['squared_error', 'absolute_error'],
    'max_features': ['auto', 'sqrt', 'log2'],
    'n_estimators': [10, 100]
}
grid = GridSearchCV(RandomForestRegressor(), param_grid, refit=True, verbose=3, n_jobs=-1)
grid.fit(X_train, y_train.values.ravel())

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_p

GridSearchCV(estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'criterion': ['squared_error', 'absolute_error'],
                         'max_features': ['auto', 'sqrt', 'log2'],
                         'n_estimators': [10, 100]},
             verbose=3)

In [35]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test)
from sklearn.metrics import r2_score
r_score=r2_score(y_test,grid_predictions)
print("The R_score value for best parameter {}:".format(grid.best_params_),r_score)

The R_score value for best parameter {'criterion': 'absolute_error', 'max_features': 'sqrt', 'n_estimators': 100}: 0.8720372532289962


In [36]:
table=pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000998,0.000583,0.000000,0.000000,squared_error,auto,10,"{'criterion': 'squared_error', 'max_features':...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
1,0.000979,0.000842,0.000000,0.000000,squared_error,auto,100,"{'criterion': 'squared_error', 'max_features':...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
2,0.027453,0.000927,0.003356,0.000262,squared_error,sqrt,10,"{'criterion': 'squared_error', 'max_features':...",0.844639,0.778641,0.775813,0.808855,0.762862,0.794162,0.029398,7
3,0.232318,0.005351,0.019239,0.001915,squared_error,sqrt,100,"{'criterion': 'squared_error', 'max_features':...",0.862281,0.794640,0.804984,0.828922,0.766349,0.811436,0.032396,4
4,0.025054,0.001065,0.003127,0.000061,squared_error,log2,10,"{'criterion': 'squared_error', 'max_features':...",0.834935,0.777364,0.814360,0.814805,0.747150,0.797723,0.031386,5
5,0.240794,0.017239,0.017799,0.001018,squared_error,log2,100,"{'criterion': 'squared_error', 'max_features':...",0.859703,0.788600,0.806476,0.832111,0.772456,0.811869,0.031054,2
6,0.000448,0.000049,0.000000,0.000000,absolute_error,auto,10,"{'criterion': 'absolute_error', 'max_features'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
7,0.000666,0.000547,0.000000,0.000000,absolute_error,auto,100,"{'criterion': 'absolute_error', 'max_features'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
8,0.082270,0.024931,0.004565,0.002577,absolute_error,sqrt,10,"{'criterion': 'absolute_error', 'max_features'...",0.858388,0.770873,0.774751,0.770098,0.764014,0.787625,0.035548,8
9,0.698300,0.016063,0.017530,0.002282,absolute_error,sqrt,100,"{'criterion': 'absolute_error', 'max_features'...",0.866286,0.791297,0.813342,0.822599,0.770722,0.812849,0.032218,1


In [37]:
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
smoker_yes_input=int(input("Smoker Yes 0 or 1:"))

In [38]:
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])

#␣change the paramter,play with it.
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[44359.97233605]
